In [ ]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta
import gzip
from pathlib import Path


# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl
import os
from pathlib import Path

In [ ]:
from __future__ import annotations


def devsig_stream_stats_v12_exporter(
    input_path: str,
    *,
    # === outputs ===
    output_onefile_jsonl: str = "ARBITRAGE/onefile.jsonl",
    output_summary_csv: str = "ARBITRAGE/summary.csv",
    output_best_params_jsonl: str = "ARBITRAGE/best_params.jsonl",
    # === thresholds ===
    # dev_sig is ALREADY in units of the ticker's own sigma, so these two are sigma multiples for every ticker (user,
    # 2026-09-21: an event starts at 0.4 sigma and exits at 0.1; RAISED to 0.6 / 0.2 on 2026-11-08). They are
    # compared with |dev_sig| exactly as it is.
    #
    # Raising BOTH together is deliberate and they are not independent: norm_thr is the EXIT, so widening the
    # entry while leaving the exit at 0.1 would stretch every event's life and change what 'normalised' means,
    # not just how many events there are. The gamma grid needs no edit - it already starts AT dev_thr
    # (bisect_left(GAMMA_LEVELS, DEV_THR...)), so its floor follows the entry automatically.
    dev_thr: float = 0.60,      # START: |dev_sig| >= dev_thr  (0.4->0.6 / 0.1->0.2 on 2026-11-08, the operator's own instruction)
    norm_thr: float = 0.20,     # EXIT / HARD: |dev_sig| <= norm_thr
    # AN EVENT IS: start (dev_thr) -> exit (norm_thr). When it reaches the exit it is CLOSED there and the next
    # divergence that reaches dev_thr is a NEW event (same class, same window). False = the old model: an event
    # lives to the end of its window and normalisation only labels it.
    close_on_exit: bool = True,
    # WHERE GAMMA READS THE RESULT. "event" = at the END of the event (its exit print, or the last print inside the
    # window if it never exited): with several events a night the old fixed 09:30 reading would score a 22:00 event
    # against a price 11 hours and two other events later. "open" = the old rule, the stack at the 09:30 snapshot
    # (PRE only; meaningful together with close_on_exit=False).
    gamma_read: str = "event",
    soft_ratio: float = 3.0,    # SOFT: abs(dev_sig) <= peak_abs / soft_ratio
    # NOISE GUARD. An event used to be born the instant ONE tick showed |dev_sig| >= dev_thr, so a
    # single bad print (a stale quote, a one-tick spike that is gone on the next row) opened an event,
    # was scored as hard/soft/none like a real one and dragged every rate. min_hold_ticks=2 requires
    # that many CONSECUTIVE ticks of this ticker, inside the class's birth window, at or beyond
    # dev_thr with the SAME sign; the event is born on the confirming tick (entry at the price
    # you could actually have acted on - the first tick is only a candidate). A tick below the
    # threshold, of the other sign, outside the window, or a new session day breaks the run. 1 = the
    # old behaviour. Runs that break before confirming are counted and printed as rejected blips.
    min_hold_ticks: int = 2,
    # THE MODEL (user, 2026-09-21): every deviation that appears in a class's birth window is TAKEN - PRE from
    # 21:00 - and HELD to the class's own close (09:30 for PRE) with no exit in between: normalisation only
    # labels the event hard/soft/none, it does not close it, and nothing is cut at 00:00 or 04:00. The ONLY thing
    # thrown away is `ignore_windows`, ALWAYS: rows inside them are dropped before the loop, so no situation can
    # start there, and a dropped tick does not count toward min_hold_ticks. Minutes, both ends inclusive; a window
    # only matters to a class whose own window contains it (in practice PRE).
    ignore_windows=(((3, 58), (4, 2)), ((7, 58), (8, 2))),
    # DISCONTINUITY GUARD (now OFF by default, see below). Measured on the published trade logs (2026-09-20): PairFlux exits pile up
    # at exactly 00:00 (1183 vs 15 at 23:59, and 99.8% of them "win"), Arbitrage births at 00:00 are
    # 14x the minute before (1445 vs 106), and births/exits spike again right after 04:00 (04:06: 996
    # vs ~50 a minute). A market does not do that on the clock - the feed's Stack% baseline / session
    # rolls over there, so a "deviation" appears or vanishes mechanically. No event may be BORN in the
    # disc_guard_min minutes after any of these clock times, and a run of ticks never spans one.
    # (Events that are ALIVE across a boundary are only flagged - `spans_disc` - not dropped.)
    # OFF by default (2026-09-21): an empty `discontinuities` guards nothing and flags nothing (`spans_disc` is
    # then always False). Set e.g. ((0, 0), (4, 0)) with disc_guard_min=10 to bring the old birth guard back.
    discontinuities=(),
    disc_guard_min: int = 0,
    # === reading ===
    assume_sorted: bool = True,
    parquet_use_pyarrow: bool = True,
    parquet_iter_batches: bool = True,
    parquet_batch_size: int = 1_000_000,
    csv_chunksize: int = 500_000,
    log_every_n_chunks: int = 5,
    # === numeric fields stored in data ===
    BENCH_NUM_FIELD: str = "Bench%",
    STOCK_NUM_FIELD: str = "Stack%",
    # === global filter for ALL outputs ===
    min_events_per_ticker: int = 10,
):
    """
    v13 exporter: PRE / OPEN / INTRA / POST only, everything split long vs short.

    BLUE, ARK, PRINT and the GLOBAL priority selector are removed. There are now
    four fully independent, symmetric event classes. Each is born, peaks, and
    resolves entirely inside its own window - none of them mute or feed each other:

      - PRE   21:00 (previous day) -> 09:30, wraps midnight. Births only until
              09:00 (PRE_START_TO) - a birth with no room left before the window
              closes can only ever resolve "none", which used to understate the
              PRE rate by ~18pp (measured on the pre-v13 onefile). NOTE that OPEN
              still begins at 09:20, so 09:00-09:20 now belongs to no class and
              takes no births at all.
      - OPEN  09:20 -> 10:00. Independent trigger, peak, hard/soft.
      - INTRA 09:45 -> 16:00.
      - POST  16:00 -> 20:00.

    Every class shares identical mechanics:
      - born when |dev_sig| >= dev_thr (0.6 sigma - dev_sig is in sigma units) inside the class's own window,
      - peak = max |dev_sig| seen since birth, frozen the instant the window ends,
      - HARD = the EXIT: |dev_sig| <= norm_thr (0.2 sigma). With close_on_exit the event ends right there and the next
        divergence starts a NEW event, so a ticker can have several events per window,
      - SOFT: |dev_sig| <= peak_abs / soft_ratio, checked from the peak onward,
      - NONE: neither happened before the window closed.

    LONG/SHORT SPLIT: a positive dev_sig at birth (long/pos) and a negative one
    (short/neg) are never blended. Every rate, bin and headline metric is kept
    separately per sign, for every class:
      - counts/rates (total/hard/soft/none, any_rate/hard_rate/soft_rate),
      - and three PRE-only ticker-level headline stats:
          - ALPHA: the peak level a ticker reaches both often and far (argmax of
            peak x hit-rate over that ticker's event peaks), one per class and sign.
          - GAMMA: the reversal-entry level (start threshold..20pp grid) with the best
            mean forward result (entry -> the END of the event: its exit print, or the
            last print of the window if it never exited - read on the stack's OWN
            Stack%), one per class and sign.
          (static.alpha / static.gamma stay the PRE ones, as the bridge reads them; the
           other classes are under top-level "levels".)
          - DELTA: the median dev_sig actually observed at the 09:30 open
            snapshot, over the last 5 days that had one, one per sign.

    OUTPUTS (3 files per run, one line/row per instrument):
      - onefile.jsonl(.gz): per-ticker rates for every class, split pos/neg.
      - summary.csv: one short row per ticker (pos/neg rates + static corr/beta/sigma).
      - best_params.jsonl(.gz): per-ticker ratings + alpha/gamma/delta levels and
        hard/soft shares, all split pos/neg.

    STRIPPED 2026-11-23 (the operator): sigma/bench bins, start/norm time bands, TOP WINDOWS,
    hard-normalisation delay, mean peak when normalised, the recent-10-day block, the last-3
    examples and the per-class event buffers are no longer computed at all. What is left is the
    ratings and the levels. DELTA still needs the per-night 09:30 reading, so pre_night and the
    recent_by_day["print"] record stay - they are inputs to a level, not outputs of their own.

    PERFORMANCE (kept from v12, no semantic changes):
      - gzip outputs: .jsonl -> .jsonl.gz (if path endswith .gz)
      - vectorized dt/dev parsing + ignored window filtering before the loop
      - optional pyarrow iter_batches()/read_row_group() for parquet
    """
    import gc, json, time, gzip, bisect
    from collections import deque, Counter
    from datetime import datetime, timedelta
    import numpy as np
    import pandas as pd


    # ---------------- gzip-aware open ----------------
    def _open_text(path: str, mode: str = "wt"):
        if str(path).lower().endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    from pathlib import Path

    Path(output_onefile_jsonl).parent.mkdir(parents=True, exist_ok=True)
    Path(output_summary_csv).parent.mkdir(parents=True, exist_ok=True)
    Path(output_best_params_jsonl).parent.mkdir(parents=True, exist_ok=True)

    onefile_f = _open_text(output_onefile_jsonl, "wt")

    CLASSES = ("pre", "open", "intra", "post")
    SIGNS = ("pos", "neg")

    summary_cols = ["ticker", "bench", "events_total"]
    summary_cols += [f"events_{c}_total" for c in CLASSES]
    for c in CLASSES:
        for sk in SIGNS:
            for m in ("any_rate", "hard_rate", "soft_rate"):
                summary_cols.append(f"{c}_{sk}_{m}")
    summary_cols += ["corr", "beta", "sigma"]

    pd.DataFrame(columns=summary_cols).to_csv(output_summary_csv, index=False, mode="w")

    best_params_f = _open_text(output_best_params_jsonl, "wt")
    best_params_f.write(json.dumps({
        "meta": {"version": "v15-ratings-levels-only", "generated_at": datetime.utcnow().isoformat() + "Z"}
    }, ensure_ascii=False) + "\n")

    # ---------------- helpers ----------------
    def _json_safe(x):
        if x is None:
            return None
        if isinstance(x, (np.floating, float)):
            if np.isnan(x) or np.isinf(x):
                return None
            return float(x)
        if isinstance(x, (np.integer, int)):
            return int(x)
        if isinstance(x, (np.bool_, bool)):
            return bool(x)
        if isinstance(x, (pd.Timestamp,)):
            return x.isoformat()
        if isinstance(x, (datetime,)):
            return x.isoformat()
        return x

    def is_finite_num(x) -> bool:
        try:
            return np.isfinite(float(x))
        except Exception:
            return False

    def in_range(t, a, b):
        return (t is not None) and (a <= t <= b)

    def in_range_wrap(t, a, b):
        """Like in_range, but supports windows that cross midnight (a > b), e.g. 21:00->09:30."""
        if t is None:
            return False
        if a <= b:
            return a <= t <= b
        return t >= a or t <= b

    def _dt_iso(x):
        return x.isoformat() if isinstance(x, datetime) else None

    # ---------------- time windows ----------------
    # PRE: pre-market class - own window, wraps midnight, 21:00 previous day -> 09:30 next
    # day. Stops at the open by design: what the spread does after 09:30 is OPEN's question.
    PRE_FROM = (21, 0)
    PRE_TO   = (9, 30)         # normalization window end / peak freeze
    # New events may only be BORN before this. Without a cutoff the birth gate and the
    # normalization gate were the same instant: an event born at 09:29 had one minute left
    # and could only ever score "none" - measured on the pre-v13 onefile that was 41 094 of
    # 123 946 PRE events (33.2%), understating the PRE rate by 18.1pp.
    # 09:00, NOT 09:20 (the operator's own instruction, 2026-11-08). KNOW WHAT THIS OPENS: OPEN's
    # own window still begins at 09:20 (OPEN_FROM), so 09:00-09:20 is now a HOLE - a deviation that
    # first reaches dev_thr in those twenty minutes is born into no class at all and is simply never
    # taken. Before this change the two windows met exactly and every pre-open minute belonged to
    # one of them. That is a deliberate narrowing, not an oversight; move OPEN_FROM to (9, 0) as well
    # if the intent was to keep them contiguous.
    PRE_START_TO = (9, 0)
    # Snapshot instant: what the spread FACTUALLY was at the open, recorded whether or not
    # it had normalized by then - so "did not come back by 09:30" is a measured value rather
    # than an absence. Read-only: it never changes PRE's HARD/SOFT status. This is also the
    # source of DELTA (median dev_sig at the open over the last 5 days, per sign).
    PRE_AT_OPEN = (9, 30)
    # Grace for the snapshot only, bounded so it cannot affect HARD/SOFT (which stopped at
    # PRE_TO). A ticker with no print in the 09:30 minute would otherwise lose the reading.
    PRE_AT_OPEN_TO = (9, 40)

    OPEN_FROM  = (9, 20)
    OPEN_TO    = (10, 0)

    INTRA_FROM = (9, 45)
    INTRA_TO   = (16, 0)

    POST_FROM  = (16, 0)
    POST_TO    = (20, 0)

    WINDOWS = {"pre": (PRE_FROM, PRE_TO), "open": (OPEN_FROM, OPEN_TO),
               "intra": (INTRA_FROM, INTRA_TO), "post": (POST_FROM, POST_TO)}
    # Birth window: PRE stops taking new births 10 minutes before its own window closes
    # (see PRE_START_TO above). OPEN/INTRA/POST take births anywhere in their own window.
    BIRTH_WINDOWS = {"pre": (PRE_FROM, PRE_START_TO), "open": (OPEN_FROM, OPEN_TO),
                      "intra": (INTRA_FROM, INTRA_TO), "post": (POST_FROM, POST_TO)}
    WRAP = {"pre": True, "open": False, "intra": False, "post": False}

    # GAMMA: candidate reversal levels, swept in the same percent units as dev. For every
    # level the event actually reached we record the stack's OWN Stack% at that moment, and
    # the result is what the stack did from there to the 09:30 reading - signed so that
    # turning against the move is the profitable side. Gamma is then the level whose mean
    # result is highest for this ticker, computed separately for pos and neg births.
    # NB: every level below (grid, alpha peaks) is in dev_sig units, i.e. sigma multiples - "PP" in the names is
    # historical. Only the RESULT of a gamma trade is a price move: Stack% percentage points.
    GAMMA_TO_PP = 20.0
    GAMMA_FINE_TO_PP = 3.0
    GAMMA_FINE_STEP_PP = 0.05
    GAMMA_COARSE_STEP_PP = 0.25
    GAMMA_MAX_ABS_PP = 25.0

    ALPHA_MAX_PEAK_PP = 25.0
    ALPHA_MIN_N = 10
    ALPHA_MIN_HIT = 0.25
    GAMMA_MIN_N = 10

    def _gamma_grid():
        # The grid is the same for every ticker; each event only uses the levels at or above its OWN start
        # threshold (0.4 sigma differs per ticker), so it starts at the finest step.
        out = []
        v = GAMMA_FINE_STEP_PP
        while v <= GAMMA_FINE_TO_PP + 1e-9:
            out.append(round(v, 6))
            v += GAMMA_FINE_STEP_PP
        v = GAMMA_FINE_TO_PP + GAMMA_COARSE_STEP_PP
        while v <= GAMMA_TO_PP + 1e-9:
            out.append(round(v, 6))
            v += GAMMA_COARSE_STEP_PP
        return out

    GAMMA_LEVELS = _gamma_grid()

    # ignored windows (known data gaps), unrelated to any specific class
    IGNORE_WINDOWS = [((int(a[0]), int(a[1])), (int(b[0]), int(b[1]))) for a, b in ignore_windows]


    # ---------------- per-class accumulator factories ----------------
    # ---------------- per-ticker state ----------------
    cur_ticker = None
    cur_day = None

    bench_name_seen = None
    static_triplet_set = False
    corr_static = beta_static = sigma_static = None

    # ALPHA/GAMMA accumulators (across this ticker's events of each class), split pos/neg
    alpha_peaks = {c: {"pos": [], "neg": []} for c in CLASSES}
    gamma_sum = {c: {"pos": {}, "neg": {}} for c in CLASSES}
    gamma_cnt = {c: {"pos": {}, "neg": {}} for c in CLASSES}

    # counts/rates split pos/neg per class - this is the "rating of total situations"
    counts = {c: {"pos": Counter(), "neg": Counter()} for c in CLASSES}

    DEV_THR = float(dev_thr)
    NORM_THR = float(norm_thr)

    def _new_pre_night():
        return {"key": None, "day_str": None, "first_sign": 0, "peak_abs": 0.0, "peak_signed": 0.0, "snap": None}

    # DELTA needs the deviation at the 09:30 open of a night that had a divergence. That used to be read off the one PRE
    # event that lived all night; events now end at their exit, so it is tracked per NIGHT instead: the sign is the
    # sign of the night's FIRST divergence (exactly what the single old event carried).
    pre_night = _new_pre_night()

    # RECENT by DAYS (PRE-only: last open-snapshot and peak per day, per sign) - source of DELTA
    recent_days = deque(maxlen=10)
    recent_by_day = {}
    last5_print_days_pos = deque(maxlen=5)
    last5_print_days_neg = deque(maxlen=5)

    def reset_ticker_state():
        nonlocal bench_name_seen, static_triplet_set, corr_static, beta_static, sigma_static
        nonlocal alpha_peaks, gamma_sum, gamma_cnt, pre_night
        nonlocal counts
        nonlocal recent_days, recent_by_day, last5_print_days_pos, last5_print_days_neg

        bench_name_seen = None
        static_triplet_set = False
        corr_static = beta_static = sigma_static = None
        alpha_peaks = {c: {"pos": [], "neg": []} for c in CLASSES}
        gamma_sum = {c: {"pos": {}, "neg": {}} for c in CLASSES}
        gamma_cnt = {c: {"pos": {}, "neg": {}} for c in CLASSES}
        pre_night = _new_pre_night()

        counts = {c: {"pos": Counter(), "neg": Counter()} for c in CLASSES}

        recent_days = deque(maxlen=10)
        recent_by_day = {}
        last5_print_days_pos = deque(maxlen=5)
        last5_print_days_neg = deque(maxlen=5)

    # ---------------- common utils ----------------


    def class_rates(counter: Counter):
        total = int(sum(counter.values()))
        hard = int(counter.get("hard", 0))
        soft = int(counter.get("soft", 0))
        none = int(counter.get("none", 0))
        any_ = hard + soft
        return {
            "total": total, "hard": hard, "soft": soft, "none": none,
            "rate_any": (any_ / total) if total else None,
            "rate_hard": (hard / total) if total else None,
            "rate_soft": (soft / total) if total else None,
            "hard_share_in_norm": (hard / (hard + soft)) if (hard + soft) else None,
        }

    def _ensure_recent_day(day_str: str):
        if day_str not in recent_by_day:
            recent_by_day[day_str] = {"print": None}
        if (len(recent_days) == 0) or (recent_days[0] != day_str):
            if day_str in recent_days:
                recent_days.remove(day_str)
            recent_days.appendleft(day_str)

    def _update_recent_daily_print(day_str, print_dt, print_dev, peak_abs, peak_signed, first_sign):
        snap = recent_by_day.get(day_str)
        if snap is None:
            recent_by_day[day_str] = {"print": None}
            snap = recent_by_day[day_str]
        if snap["print"] is None:
            snap["print"] = {
                "dt": _dt_iso(print_dt), "dev": _json_safe(print_dev),
                "pre_peak_abs": _json_safe(peak_abs), "pre_peak_signed": _json_safe(peak_signed),
                "first_sign": int(first_sign),
            }


    # ---------------- event state (PRE/OPEN/INTRA/POST, symmetric) ----------------
    def _default_event_state(cls):
        d = {
            "active": False,
            "start_dt": None, "start_dev": np.nan, "start_sign": 0, "start_stock": np.nan, "start_bench": np.nan,
            "peak_abs": 0.0, "peak_signed": 0.0, "peak_dt": None, "peak_stock": np.nan, "peak_bench": np.nan,
            "hard_dt": None, "hard_val": np.nan, "hard_stock": np.nan, "hard_bench": np.nan,
            "soft_found": False, "soft_dt": None, "soft_val": np.nan, "soft_stock": np.nan, "soft_bench": np.nan,
        }
        d["gam_entry_stock"] = {}     # gamma level -> Stack% at the first print that reached it
        d["last_stock"] = np.nan      # Stack% of the last print INSIDE the class window (the end reading if it never exits)
        return d

    events = {c: _default_event_state(c) for c in CLASSES}
    MIN_HOLD = max(1, int(min_hold_ticks))
    _disc_minutes = set()
    for _dh, _dm in discontinuities:
        for _k in range(int(disc_guard_min)):
            _disc_minutes.add(((_dh * 60 + _dm + _k) % 1440))
    def in_disc_guard(t):
        return (t[0] * 60 + t[1]) in _disc_minutes
    DISC_IDX = [(_dh * 60 + _dm) for _dh, _dm in discontinuities]   # on the PRE-wrapped axis (evening < 0)
    run_len = {c: 0 for c in CLASSES}    # consecutive qualifying ticks so far, per class
    run_sign = {c: 0 for c in CLASSES}
    noise = {"confirmed": 0, "blips": 0}

    def _end_run(cls):
        # a run that ended before it confirmed is one-tick (or short) noise
        if 0 < run_len[cls] < MIN_HOLD:
            noise["blips"] += 1
        run_len[cls] = 0
    event_ids = {c: 0 for c in CLASSES}  # file-level counters, kept across tickers (debug ids only)

    def reset_event(cls):
        events[cls] = _default_event_state(cls)

    def start_event(cls, dt_now, dev_now, stock_pct, bench_pct):
        e = _default_event_state(cls)
        e["active"] = True
        e["start_dt"] = dt_now
        e["start_dev"] = float(dev_now)
        e["start_sign"] = 1 if float(dev_now) >= 0 else -1
        e["start_stock"] = stock_pct
        e["start_bench"] = bench_pct

        a = abs(float(dev_now))
        e["peak_abs"] = a
        e["peak_signed"] = float(dev_now)
        e["peak_dt"] = dt_now
        e["peak_stock"] = stock_pct
        e["peak_bench"] = bench_pct

        if is_finite_num(stock_pct):
            e["last_stock"] = float(stock_pct)
            # An event is born already past its start threshold, so every gamma level from that threshold up to
            # its birth deviation was reached on this very tick. Levels below the threshold are not levels.
            for L in GAMMA_LEVELS[bisect.bisect_left(GAMMA_LEVELS, DEV_THR * (1.0 - 1e-9)):]:
                if L <= a:
                    e["gam_entry_stock"][L] = float(stock_pct)
                else:
                    break

        if cls == "pre":
            _note_pre_birth(dt_now, e["start_sign"])

        events[cls] = e

    def process_tick(cls, dt_now, dev_now, stock_pct, bench_pct, t):
        e = events[cls]
        cur_abs = abs(float(dev_now))

        # gamma entries are gated at the class's BIRTH window (09:20 for PRE): an entry with no time left to
        # work is the same bug that once cost PRE 18pp of rate.
        bf, bt = BIRTH_WINDOWS[cls]
        if (in_range_wrap(t, bf, bt) if WRAP[cls] else in_range(t, bf, bt)) and is_finite_num(stock_pct):
            _i0 = bisect.bisect_left(GAMMA_LEVELS, DEV_THR * (1.0 - 1e-9))
            for L in GAMMA_LEVELS[_i0:]:
                if L > cur_abs:
                    break
                if L not in e["gam_entry_stock"]:
                    e["gam_entry_stock"][L] = float(stock_pct)

        wf, wt = WINDOWS[cls]
        inside = in_range_wrap(t, wf, wt) if WRAP[cls] else in_range(t, wf, wt)
        if not inside:
            return

        if is_finite_num(stock_pct):
            e["last_stock"] = float(stock_pct)

        if cur_abs > float(e["peak_abs"]):
            e["peak_abs"] = float(cur_abs)
            e["peak_signed"] = float(dev_now)
            e["peak_dt"] = dt_now
            e["peak_stock"] = stock_pct
            e["peak_bench"] = bench_pct
            if e["hard_dt"] is None:
                e["soft_found"] = False
                e["soft_dt"] = None
                e["soft_val"] = np.nan
                e["soft_stock"] = np.nan
                e["soft_bench"] = np.nan

        if e["hard_dt"] is None and cur_abs <= NORM_THR:
            e["hard_dt"] = dt_now
            e["hard_val"] = float(dev_now)
            e["hard_stock"] = stock_pct
            e["hard_bench"] = bench_pct
            if close_on_exit:
                # The exit ends the event on this very print; the class is free for the next divergence.
                finalize_event(cls, reason="exit")
                return

        if e["hard_dt"] is None and isinstance(e["peak_dt"], datetime) and e["peak_abs"] > 0:
            if dt_now >= e["peak_dt"] and cur_abs <= (float(e["peak_abs"]) / float(soft_ratio)):
                if (not e["soft_found"]) or (dt_now < e["soft_dt"]):
                    e["soft_found"] = True
                    e["soft_dt"] = dt_now
                    e["soft_val"] = float(dev_now)
                    e["soft_stock"] = stock_pct
                    e["soft_bench"] = bench_pct

    def finalize_event(cls, reason="window_end"):
        e = events[cls]
        if not e["active"]:
            return

        if e["hard_dt"] is not None:
            status = "hard"
            end_dt = e["hard_dt"]
            end_val, end_stock, end_bench = e["hard_val"], e["hard_stock"], e["hard_bench"]
        elif e["soft_found"] and e["soft_dt"] is not None:
            status = "soft"
            end_dt = e["soft_dt"]
            end_val, end_stock, end_bench = e["soft_val"], e["soft_stock"], e["soft_bench"]
        else:
            status = "none"
            end_dt = None
            end_val = end_stock = end_bench = np.nan

        sk = "pos" if e["start_sign"] > 0 else "neg"

        counts[cls][sk][status] += 1

        # ALPHA sample: this event's own peak, whether or not it exited. Split by the event's own sign.
        if is_finite_num(e["peak_abs"]) and 0 < abs(float(e["peak_abs"])) <= ALPHA_MAX_PEAK_PP:
            alpha_peaks[cls][sk].append(abs(float(e["peak_abs"])))

        # GAMMA samples: one per level this event reached, accumulated into this event's own sign bucket so a
        # "gap up, fade" ticker and a "gap down, bounce" ticker never share a gamma level. The result is read on
        # the STACK at the END of the event: the exit print, or the last print inside the window if it never exited.
        _end_stock = e["hard_stock"] if e["hard_dt"] is not None else e["last_stock"]
        if str(gamma_read).lower() == "open" and cls == "pre" and pre_night["snap"] is not None:
            _end_stock = pre_night["snap"]["stock"]
        if e["gam_entry_stock"] and is_finite_num(_end_stock):
            _sf = -1.0 if sk == "pos" else 1.0
            for _L, _entry_stock in e["gam_entry_stock"].items():
                _res = _sf * (float(_end_stock) - float(_entry_stock))
                if abs(_res) > GAMMA_MAX_ABS_PP:
                    continue
                gamma_sum[cls][sk][_L] = gamma_sum[cls][sk].get(_L, 0.0) + _res
                gamma_cnt[cls][sk][_L] = gamma_cnt[cls][sk].get(_L, 0) + 1

        if cls == "pre":
            # DELTA source (per night): the night's peak so far, for the daily peak record.
            if is_finite_num(e["peak_abs"]) and float(e["peak_abs"]) > pre_night["peak_abs"]:
                pre_night["peak_abs"] = float(e["peak_abs"])
                pre_night["peak_signed"] = float(e["peak_signed"])
            if isinstance(e["start_dt"], datetime):
                # Still registers the day: recent_days is a maxlen=10 deque and which days are in it
                # decides what DELTA reads, so dropping this would move DELTA.
                _ensure_recent_day(pre_night["day_str"] or e["start_dt"].date().isoformat())
            _recompute_last5()


        event_ids[cls] += 1
        reset_event(cls)

    def _recompute_last5():
        last5_print_days_pos.clear(); last5_print_days_neg.clear()
        for d in list(recent_days):
            rsnap = recent_by_day.get(d)
            if not rsnap:
                continue
            pr = rsnap.get("print")
            if pr and is_finite_num(pr.get("dev")):
                (last5_print_days_pos if int(pr.get("first_sign", 1)) > 0 else last5_print_days_neg).append(float(pr["dev"]))
            if len(last5_print_days_pos) >= 5 and len(last5_print_days_neg) >= 5:
                break

    def _note_pre_birth(dt_now, sign):
        if pre_night["first_sign"] == 0:
            pre_night["first_sign"] = int(sign)
            pre_night["day_str"] = dt_now.date().isoformat()

    def _pre_night_tick(dt_now, t, dev_now, stock_pct):
        """Roll the per-night PRE record over and take the 09:30 open reading (DELTA) once per night."""
        nonlocal pre_night
        if in_range_wrap(t, PRE_FROM, PRE_TO):
            d0 = dt_now.date()
            key = (d0 + timedelta(days=1)).isoformat() if t >= PRE_FROM else d0.isoformat()
            if key != pre_night["key"]:
                pre_night = _new_pre_night()
                pre_night["key"] = key
        if (pre_night["snap"] is None and pre_night["first_sign"] != 0 and in_range(t, PRE_AT_OPEN, PRE_AT_OPEN_TO)):
            cur_abs = abs(float(dev_now))
            pre_night["snap"] = {
                "dt": dt_now, "signed": float(dev_now), "abs": float(cur_abs),
                "normalized": bool(cur_abs <= NORM_THR),
                "stock": float(stock_pct) if is_finite_num(stock_pct) else np.nan,
            }
            e_pre = events["pre"]
            peak_abs, peak_signed = pre_night["peak_abs"], pre_night["peak_signed"]
            if e_pre["active"] and float(e_pre["peak_abs"]) > peak_abs:
                peak_abs, peak_signed = float(e_pre["peak_abs"]), float(e_pre["peak_signed"])
            day_str = pre_night["day_str"]
            _ensure_recent_day(day_str)
            _update_recent_daily_print(day_str, dt_now, float(dev_now), peak_abs, peak_signed, int(pre_night["first_sign"]))
            _recompute_last5()

    def on_new_day():
        for _c in CLASSES:
            _end_run(_c)
        e_pre = events["pre"]
        if e_pre["active"]:
            start_t = (e_pre["start_dt"].hour, e_pre["start_dt"].minute) if isinstance(e_pre["start_dt"], datetime) else None
            started_overnight = start_t is not None and start_t >= PRE_FROM
            if not started_overnight:
                finalize_event("pre", reason="day_boundary")
        for cls in ("open", "intra", "post"):
            if events[cls]["active"]:
                finalize_event(cls, reason="day_boundary")


    def median_or_none(arr):
        arr = list(arr)
        return _json_safe(float(np.median(arr)) if arr else None)

    def _compute_alpha(peaks):
        n = len(peaks)
        if n < ALPHA_MIN_N:
            return None, None, n
        pk = np.sort(np.asarray(peaks, dtype=float))[::-1]
        k = np.arange(1, pk.size + 1, dtype=float)
        hit_ok = (k / pk.size) >= ALPHA_MIN_HIT
        if not hit_ok.any():
            return None, None, n
        i = int(np.argmax(np.where(hit_ok, pk * k, -np.inf)))
        return float(pk[i]), float(k[i] / pk.size), n

    def _compute_gamma(sum_map, cnt_map):
        best = None
        for L in sorted(cnt_map):
            n = int(cnt_map[L])
            if n < GAMMA_MIN_N:
                continue
            m = float(sum_map[L]) / n
            if best is None or m > best[0]:
                best = (m, float(L), n)
        if best is not None and best[0] > 0.0:
            return best[1], best[0], best[2]  # level, expected_value, n
        return None, None, 0

    # ---------------- flush ticker (write files) ----------------
    def flush_current_ticker():
        nonlocal cur_ticker, cur_day
        nonlocal bench_name_seen, corr_static, beta_static, sigma_static

        if cur_ticker is None:
            return

        for _c in CLASSES:
            _end_run(_c)
        for cls in CLASSES:
            if events[cls]["active"]:
                finalize_event(cls, reason="ticker_end")

        # ALPHA/GAMMA for this ticker: one value per class and sign; DELTA (PRE only) one per sign
        levels = {}
        for c in CLASSES:
            levels[c] = {"alpha": {}, "gamma": {}}
            for sk in SIGNS:
                a_val, a_hit, a_n = _compute_alpha(alpha_peaks[c][sk])
                levels[c]["alpha"][sk] = {"value": _json_safe(a_val), "hit": _json_safe(a_hit), "n": a_n}
                g_lvl, g_ev, g_n = _compute_gamma(gamma_sum[c][sk], gamma_cnt[c][sk])
                levels[c]["gamma"][sk] = {"level": _json_safe(g_lvl), "ev": _json_safe(g_ev), "n": g_n}
        alpha = levels["pre"]["alpha"]      # what the bridge reads as static.alpha / static.gamma: the PRE ones
        gamma = levels["pre"]["gamma"]
        delta = {
            "pos": median_or_none(last5_print_days_pos),
            "neg": median_or_none(last5_print_days_neg),
        }

        rates = {c: {sk: class_rates(counts[c][sk]) for sk in SIGNS} for c in CLASSES}
        events_totals = {c: rates[c]["pos"]["total"] + rates[c]["neg"]["total"] for c in CLASSES}
        events_total = sum(events_totals.values())

        if events_total < int(min_events_per_ticker):
            reset_ticker_state()
            return

        payload = {
            "ticker": cur_ticker,
            "bench": bench_name_seen,
            "static": {
                "corr": _json_safe(corr_static), "beta": _json_safe(beta_static), "sigma": _json_safe(sigma_static),
                "alpha": alpha, "gamma": gamma, "delta": delta,
            },
            "levels": levels,
            "params": {
                "dev_thr": float(dev_thr), "norm_thr": float(norm_thr), "soft_ratio": float(soft_ratio),
                "close_on_exit": bool(close_on_exit),
                "gamma_read": str(gamma_read), "min_hold_ticks": MIN_HOLD,
                "windows": {
                    "pre": "21:00(prev day)-09:30, wraps midnight; births until 09:20; own peak frozen at 09:30",
                    "open": "09:20-10:00; independent trigger/peak/hard/soft",
                    "intra": "09:45-16:00",
                    "post": "16:00-20:00",
                    "ignored_gaps": ["%02d:%02d-%02d:%02d" % (a[0], a[1], b[0], b[1]) for a, b in IGNORE_WINDOWS],
                },
                "min_events_per_ticker": int(min_events_per_ticker),
            },
            "stats": {
                "events_total": int(events_total),
                **{
                    c: {
                        "events_total": events_totals[c],
                        "rates": rates[c],  # {"pos": {...}, "neg": {...}}
                    } for c in CLASSES
                },
            },
        }

        onefile_f.write(json.dumps(payload, ensure_ascii=False) + "\n")

        row = {"ticker": cur_ticker, "bench": bench_name_seen, "events_total": int(events_total)}
        row.update({f"events_{c}_total": events_totals[c] for c in CLASSES})
        for c in CLASSES:
            for sk in SIGNS:
                row[f"{c}_{sk}_any_rate"] = _json_safe(rates[c][sk]["rate_any"])
                row[f"{c}_{sk}_hard_rate"] = _json_safe(rates[c][sk]["rate_hard"])
                row[f"{c}_{sk}_soft_rate"] = _json_safe(rates[c][sk]["rate_soft"])
        row["corr"] = _json_safe(corr_static)
        row["beta"] = _json_safe(beta_static)
        row["sigma"] = _json_safe(sigma_static)
        pd.DataFrame([row], columns=summary_cols).to_csv(output_summary_csv, mode="a", header=False, index=False)

        best = {
            "ticker": cur_ticker,
            "bench": bench_name_seen,
            "static": {
                "corr": _json_safe(corr_static), "beta": _json_safe(beta_static), "sigma": _json_safe(sigma_static),
                "alpha": alpha, "gamma": gamma, "delta": delta,
            },
            "levels": levels,
            "totals": {"events_total": int(events_total), **{f"{c}_total": events_totals[c] for c in CLASSES}},
            "ratings": {c: {sk: _json_safe(rates[c][sk]["rate_any"]) for sk in SIGNS} for c in CLASSES},
            "hard_soft_share": {
                c: {sk: {"hard": int(rates[c][sk]["hard"]), "soft": int(rates[c][sk]["soft"]),
                         "hard_share": _json_safe(rates[c][sk]["hard_share_in_norm"])} for sk in SIGNS}
                for c in CLASSES
            },
            "params": {"dev_thr": float(dev_thr), "norm_thr": float(norm_thr), "soft_ratio": float(soft_ratio),
                       "close_on_exit": bool(close_on_exit),
                       "min_hold_ticks": MIN_HOLD},
        }

        best_params_f.write(json.dumps(best, ensure_ascii=False) + "\n")
        reset_ticker_state()

    # ---------------- processing chunks ----------------
    def process_chunk(chunk: "pd.DataFrame", ci: int):
        nonlocal cur_ticker, cur_day
        nonlocal bench_name_seen, static_triplet_set, corr_static, beta_static, sigma_static

        req = {"ticker", "date", "dt", "dev_sig"}
        if not req.issubset(chunk.columns):
            raise KeyError(f"Input must contain columns: {sorted(req)}")

        if not assume_sorted:
            chunk["dt"] = pd.to_datetime(chunk["dt"], errors="coerce", utc=True)
            chunk.sort_values(["ticker", "date", "dt"], inplace=True)

        def col(name):
            return chunk[name] if name in chunk.columns else pd.Series(np.nan, index=chunk.index)

        s_ticker = col("ticker")
        s_date = col("date")
        s_dt_ts = pd.to_datetime(col("dt"), errors="coerce", utc=True)
        s_dev = pd.to_numeric(col("dev_sig"), errors="coerce")

        s_bench_name = col("bench")
        s_corr = col("corr")
        s_beta = col("beta")
        s_sigma = col("sigma")

        s_bench_num = (pd.to_numeric(col(BENCH_NUM_FIELD), errors="coerce")
                       if BENCH_NUM_FIELD in chunk.columns else pd.Series(np.nan, index=chunk.index))
        s_stock_pct = (pd.to_numeric(col(STOCK_NUM_FIELD), errors="coerce")
                       if STOCK_NUM_FIELD in chunk.columns else pd.Series(np.nan, index=chunk.index))

        dt_ok = s_dt_ts.notna()
        dev_ok = np.isfinite(s_dev.to_numpy(dtype="float64", copy=False))
        mask = (dt_ok.to_numpy(copy=False) & dev_ok)

        if mask.any():
            dt2 = s_dt_ts[mask]
            h = dt2.dt.hour.to_numpy(dtype="int16", copy=False)
            m = dt2.dt.minute.to_numpy(dtype="int16", copy=False)

            def _in_win(h, m, a, b):
                ah, am = a
                bh, bm = b
                return ((h > ah) | ((h == ah) & (m >= am))) & ((h < bh) | ((h == bh) & (m <= bm)))

            ig = np.zeros_like(h, dtype=bool)
            for a, b in IGNORE_WINDOWS:
                ig |= _in_win(h, m, a, b)

            idx_mask = np.flatnonzero(mask)
            mask[idx_mask] = ~ig

        if not mask.any():
            return

        dt_py = s_dt_ts[mask].dt.to_pydatetime()
        tk_arr = s_ticker[mask].to_numpy(copy=False)
        ds_arr = s_date[mask].to_numpy(copy=False)
        dev_arr = s_dev[mask].to_numpy(dtype="float64", copy=False)
        bench_arr = s_bench_num[mask].to_numpy(dtype="float64", copy=False)
        stock_arr = s_stock_pct[mask].to_numpy(dtype="float64", copy=False)

        bench_name_arr = (s_bench_name[mask].to_numpy(copy=False) if "bench" in chunk.columns else None)
        corr_arr = (s_corr[mask].to_numpy(copy=False) if "corr" in chunk.columns else None)
        beta_arr = (s_beta[mask].to_numpy(copy=False) if "beta" in chunk.columns else None)
        sigma_arr = (s_sigma[mask].to_numpy(copy=False) if "sigma" in chunk.columns else None)

        dt2 = s_dt_ts[mask]
        h_arr = dt2.dt.hour.to_numpy(dtype="int16", copy=False).tolist()
        m_arr = dt2.dt.minute.to_numpy(dtype="int16", copy=False).tolist()

        n = len(dev_arr)
        for i in range(n):
            tk = tk_arr[i]
            ds = ds_arr[i]
            dt_now = dt_py[i]
            dev_now = dev_arr[i]
            bench_num = bench_arr[i]
            stock_pct = stock_arr[i]
            t = (int(h_arr[i]), int(m_arr[i]))

            if cur_ticker is not None and tk != cur_ticker:
                flush_current_ticker()
                cur_ticker, cur_day = tk, ds
                on_new_day()

            if cur_ticker is None:
                cur_ticker, cur_day = tk, ds
                on_new_day()

            if bench_name_seen is None and bench_name_arr is not None:
                bn = bench_name_arr[i]
                if pd.notna(bn) and str(bn).strip():
                    bench_name_seen = str(bn)

            if (not static_triplet_set) and (corr_arr is not None) and (beta_arr is not None) and (sigma_arr is not None):
                c = corr_arr[i]; b = beta_arr[i]; s = sigma_arr[i]
                if pd.notna(c) and pd.notna(b) and pd.notna(s):
                    corr_static, beta_static, sigma_static = c, b, s
                    static_triplet_set = True

            if ds != cur_day:
                cur_day = ds
                on_new_day()

            # PRE wraps midnight, so its "passed the open-snapshot grace window" check needs
            # the actual calendar date, not a bare (hour, minute) tuple compare.
            e_pre = events["pre"]
            if e_pre["active"] and isinstance(e_pre["start_dt"], datetime):
                _pre_start_t = (e_pre["start_dt"].hour, e_pre["start_dt"].minute)
                _close_dt = e_pre["start_dt"].replace(hour=PRE_AT_OPEN_TO[0], minute=PRE_AT_OPEN_TO[1], second=0, microsecond=0)
                if _pre_start_t > PRE_AT_OPEN_TO:
                    _close_dt = _close_dt + timedelta(days=1)
                if dt_now > _close_dt:
                    finalize_event("pre", reason="passed_open_window")

            for cls in ("open", "intra", "post"):
                if events[cls]["active"] and (t > WINDOWS[cls][1]):
                    finalize_event(cls, reason=f"passed_{cls}_window")

            for cls in CLASSES:
                if not events[cls]["active"]:
                    bf, bt = BIRTH_WINDOWS[cls]
                    born_inside = in_range_wrap(t, bf, bt) if WRAP[cls] else in_range(t, bf, bt)
                    if born_inside and (not in_disc_guard(t)) and abs(float(dev_now)) >= DEV_THR:
                        sg = 1 if float(dev_now) >= 0 else -1
                        if run_len[cls] > 0 and run_sign[cls] == sg:
                            run_len[cls] += 1
                        else:
                            _end_run(cls)          # the other sign cut the previous run short
                            run_len[cls], run_sign[cls] = 1, sg
                        if run_len[cls] >= MIN_HOLD:
                            start_event(cls, dt_now, dev_now, stock_pct, bench_num)
                            noise["confirmed"] += 1
                            run_len[cls] = 0
                    else:
                        _end_run(cls)
                else:
                    run_len[cls] = 0
                if events[cls]["active"]:
                    process_tick(cls, dt_now, dev_now, stock_pct, bench_num, t)

            _pre_night_tick(dt_now, t, dev_now, stock_pct)

    # ---------------- main read loop ----------------
    t0 = time.time()
    total_rows = 0
    last_rows = 0
    last_ts = t0

    is_parquet = str(input_path).lower().endswith((".parquet", ".pq", ".parq"))
    print(
        f"START v13 exporter (pre/open/intra/post, pos/neg split) file={input_path} parquet={is_parquet} "
        f"dev_thr={dev_thr} norm_thr={norm_thr} (sigma units) close_on_exit={close_on_exit} soft_ratio={soft_ratio} "
        f"min_events={min_events_per_ticker}"
    )

    try:
        if is_parquet and parquet_use_pyarrow:
            import pyarrow.parquet as pq
            pf = pq.ParquetFile(input_path)

            wanted = ["ticker", "date", "dt", "dev_sig", "bench", "corr", "beta", "sigma", STOCK_NUM_FIELD, BENCH_NUM_FIELD]
            cols = [c for c in wanted if c in pf.schema.names]

            for ci in range(pf.num_row_groups):
                chunk = pf.read_row_group(ci, columns=cols).to_pandas()
                process_chunk(chunk, ci + 1)
                total_rows += len(chunk)

                if (ci + 1) % log_every_n_chunks == 0:
                    now = time.time()
                    rps = (total_rows - last_rows) / max(now - last_ts, 1e-6)
                    print(f"[rg {ci+1:>4}/{pf.num_row_groups}] rows={total_rows:,} speed={rps:,.0f}/s elapsed={now-t0:,.1f}s")
                    last_rows, last_ts = total_rows, now

                del chunk
                if (ci + 1) % max(10, log_every_n_chunks * 2) == 0:
                    gc.collect()

        elif not is_parquet:
            reader = pd.read_csv(input_path, compression="infer", low_memory=False, chunksize=csv_chunksize)
            for ci, chunk in enumerate(reader, 1):
                process_chunk(chunk, ci)
                total_rows += len(chunk)

                if ci % log_every_n_chunks == 0:
                    now = time.time()
                    rps = (total_rows - last_rows) / max(now - last_ts, 1e-6)
                    print(f"[chunk {ci:>5}] rows={total_rows:,} speed={rps:,.0f}/s elapsed={now-t0:,.1f}s")
                    last_rows, last_ts = total_rows, now

                del chunk
                if ci % max(10, log_every_n_chunks * 2) == 0:
                    gc.collect()

        else:
            df = pd.read_parquet(input_path)
            step = 1_000_000
            for ci, start in enumerate(range(0, len(df), step), 1):
                chunk = df.iloc[start:start + step]
                process_chunk(chunk, ci)
                total_rows += len(chunk)

                if ci % log_every_n_chunks == 0:
                    now = time.time()
                    rps = (total_rows - last_rows) / max(now - last_ts, 1e-6)
                    print(f"[chunk {ci:>5}] rows={total_rows:,} speed={rps:,.0f}/s elapsed={now-t0:,.1f}s")
                    last_rows, last_ts = total_rows, now

                del chunk
                if ci % max(10, log_every_n_chunks * 2) == 0:
                    gc.collect()

        flush_current_ticker()
        _tot = noise["confirmed"] + noise["blips"]
        print(f"noise guard: min_hold_ticks={MIN_HOLD} events confirmed={noise['confirmed']:,} "
              f"blips rejected={noise['blips']:,} ({100.0 * noise['blips'] / _tot if _tot else 0.0:.1f}% of candidate runs)")
        print(f"DONE rows={total_rows:,} -> onefile={output_onefile_jsonl} summary={output_summary_csv} best_params={output_best_params_jsonl}")

    finally:
        onefile_f.close()
        best_params_f.close()


In [ ]:
from pathlib import Path
import os
import pandas as pd


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break

        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME (recommended).")

        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


# ------------------------------------------------------------------
# MAIN (NO SORTING)
# ------------------------------------------------------------------

FINAL_PATH, OUT_DIR = _resolve_orion_paths("arbitrage")

print("Using FINAL parquet (no sorting):", FINAL_PATH)

devsig_stream_stats_v12_exporter(
    input_path=str(FINAL_PATH),  # <-- напряму

    output_onefile_jsonl=str(OUT_DIR / "onefile.jsonl.gz"),
    output_best_params_jsonl=str(OUT_DIR / "best_params.jsonl.gz"),
    output_summary_csv=str(OUT_DIR / "summary.csv"),

    dev_thr=0.60,     # start: 0.6 sigma (dev_sig is already in sigma units)
    norm_thr=0.20,    # exit: 0.2 sigma
    close_on_exit=True,   # the exit closes the event; the next divergence is a new one
    soft_ratio=3.0,

    min_events_per_ticker=10,
)

print("Arbitrage completed (no pre-sorting).")

In [ ]:
from __future__ import annotations


def devsig_rolling_perf_exporter(
    input_path: str,
    output_path: str = "ARBITRAGE/rolling_perf.json.gz",
    *,
    windows_days=(5, 20, 40, 65),
    dev_thr: float = 0.60,      # start 0.6 sigma (dev_sig is in sigma units); exit norm_thr = 0.2
    norm_thr: float = 0.20,
    soft_ratio: float = 3.0,
    # NOISE GUARD. An event used to be born the instant ONE tick showed |dev_sig| >= dev_thr, so a
    # single bad print (a stale quote, a one-tick spike that is gone on the next row) opened an event,
    # was scored as hard/soft/none like a real one and dragged every rate. min_hold_ticks=2 requires
    # that many CONSECUTIVE ticks of this ticker, inside the class's birth window, at or beyond
    # dev_thr with the SAME sign; the event is born on the confirming tick (entry at the price
    # you could actually have acted on - the first tick is only a candidate). A tick below the
    # threshold, of the other sign, outside the window, or a new session day breaks the run. 1 = the
    # old behaviour. Runs that break before confirming are counted and printed as rejected blips.
    min_hold_ticks: int = 2,
    # THE MODEL (user, 2026-09-21): every deviation that appears in a class's birth window is TAKEN - PRE from
    # 21:00 - and HELD to the class's own close (09:30 for PRE) with no exit in between: normalisation only
    # labels the event hard/soft/none, it does not close it, and nothing is cut at 00:00 or 04:00. The ONLY thing
    # thrown away is `ignore_windows`, ALWAYS: rows inside them are dropped before the loop, so no situation can
    # start there, and a dropped tick does not count toward min_hold_ticks. Minutes, both ends inclusive; a window
    # only matters to a class whose own window contains it (in practice PRE).
    ignore_windows=(((3, 58), (4, 2)), ((7, 58), (8, 2))),
    # THE 04:02 MODEL (Scout toggle "FROM 04:02"). The published PRE trades take every deviation from 21:00 and hold it
    # to 09:30. This adds a SECOND, independent PRE track for "what if trading only STARTS at 04:02": same rule (a
    # deviation is taken the moment |dev_sig| >= dev_thr holds for min_hold_ticks, held to the same close, nothing
    # exits early) but births are only allowed from `pre_alt_from`. The rule is STATE-based, so every deviation that
    # is already open when trading starts is picked up at its 04:02 price - not at the price it started at overnight -
    # and a ticker whose overnight deviation had normalised can start a NEW one later that the 21:00 track (one event
    # per ticker and window) never sees. The trades go to the SAME trade log, flagged "alt": true, and are kept out
    # of every rate and cell statistic. None switches the track off.
    pre_alt_from=(4, 2),
    # DISCONTINUITY GUARD (now OFF by default, see below). Measured on the published trade logs (2026-09-20): PairFlux exits pile up
    # at exactly 00:00 (1183 vs 15 at 23:59, and 99.8% of them "win"), Arbitrage births at 00:00 are
    # 14x the minute before (1445 vs 106), and births/exits spike again right after 04:00 (04:06: 996
    # vs ~50 a minute). A market does not do that on the clock - the feed's Stack% baseline / session
    # rolls over there, so a "deviation" appears or vanishes mechanically. No event may be BORN in the
    # disc_guard_min minutes after any of these clock times, and a run of ticks never spans one.
    # (Events that are ALIVE across a boundary are only flagged - `spans_disc` - not dropped.)
    # OFF by default (2026-09-21): an empty `discontinuities` guards nothing and flags nothing (`spans_disc` is
    # then always False). Set e.g. ((0, 0), (4, 0)) with disc_guard_min=10 to bring the old birth guard back.
    discontinuities=(),
    disc_guard_min: int = 0,
    # bucket the START (birth) deviation - no lookahead, this is what you'd actually
    # know the instant you'd enter. dev_thr is the floor (nothing is born below it).
    start_bin_step: float = 0.2,
    start_bin_max: float = 3.0,
    min_events_per_ticker: int = 10,
    min_events_per_cell: int = 2,
    # $ position size assumed for every trade when turning an event into P&L.
    position_usd: float = 1000.0,
    # emit the raw per-event trade log (needed to draw a P&L curve for an arbitrary,
    # user-chosen "start" threshold - the bucket grid below is fixed-step and can't
    # answer "what if start=0.47" on its own).
    include_trade_log: bool = True,
    BENCH_NUM_FIELD: str = "Bench%",
    STOCK_NUM_FIELD: str = "Stack%",
    parquet_batch_log_every: int = 10,
):
    """
    Rolling recency performance, split by class x sign x start-deviation bucket x
    trailing window (default last 5/20/40/65 sessions), PLUS a raw per-event trade log
    for P&L-curve rendering. Built for the "Scout" UI: pick a ticker/class/sign/
    window, set a "start" threshold (in %, sigma, gamma- or delta-relative units -
    gamma/delta come from the companion best_params.jsonl.gz, joined by ticker),
    and get both a hit-rate and a $ P&L curve, entirely client-side against this
    one file (no backend recompute per interaction).

    WHY START, NOT PEAK: the existing v13 onefile buckets by the PEAK deviation,
    which is only known after the fact. Bucketing by the peak here would let a
    trade that started at 0.31 and later widened to 1.40 count as a "1.2-1.4"
    signal, which you could not have known at entry. Both the bucket grid AND the
    trade log key off the deviation observed the moment the class fires.

    WHY WINDOWS ARE FILE-RELATIVE: final.parquet is a static historical file, not a
    live feed, so "last 5/20/40/65 days" means the 5/20/40/65 most recent SESSION DATES
    that are actually present in the file (per the `date` column), not
    datetime.date.today(). A session's rank is 0 for the most recent date, 1 for
    the one before it, etc.; an event's start_date qualifies for window W iff its
    rank < W. Windows are cumulative (last-5 subset-of last-20 subset-of last-40).
    The trade log carries every event within the LARGEST window (default 40); the
    frontend slices it down to 5/20 using meta.recent_dates (see below) rather than
    the backend emitting three overlapping copies of the same trades.

    Classes/windows/thresholds are identical to the v13 onefile exporter (PRE
    21:00(prev day)-09:30 wraps midnight/ births until 09:20, OPEN 09:20-10:00,
    INTRA 09:45-16:00, POST 16:00-20:00; HARD <=norm_thr, SOFT <=peak/soft_ratio),
    so a ticker's rate here is directly comparable to its all-time rate there.

    P&L MODEL (real Stack% move, not a synthetic sigma-compression number):
      - entry price marker = Stack% at the tick the class was born (start_stock).
      - exit price marker  = Stack% at the LAST tick inside the class's own window -
        i.e. always held to that class's own close (09:30 PRE, 10:00 OPEN, 16:00
        INTRA, 20:00 POST), for hard/soft/none alike. Hard/soft still decide the
        rate/rating (did it normalize before the window closed?), but the $ P&L is
        always marked at the same clock time for every trade in a class, so trades
        are comparable to each other regardless of how quickly each one resolved.
      - direction: a "pos" event (dev_sig>0: the stack ran ahead of the benchmark)
        is a SHORT; a "neg" event is a LONG. This is the mean-reversion trade the
        whole class exists to describe - you fade the leg that got ahead.
      - pnl_pct = (start_stock - exit_stock) for pos [short], (exit_stock -
        start_stock) for neg [long] - both in percentage POINTS, same convention
        already used by v13's GAMMA (`_sf = -1 if pos else 1`).
      - pnl_usd = position_usd * pnl_pct / 100.

    OUTPUT: a single gzip-if-.gz-suffixed JSON file:
      {"meta": {..., "recent_dates": [...], "position_usd": 1000.0},
       "rows": [ {ticker, bench, corr, beta, sigma,
                   classes: {pre: {pos: {"0.30-0.50": {"w5": {...}, ...}}}}},
                   trades: [{cls, sign, date, start_dev, start_gap_pct, status, pnl_pct, pnl_usd,
                             start_minute_idx, peak_minute_idx, peak_dev_abs, end_dev_abs,
                             end_gap_pct}, ...] (chronological) },
                  ... ]}
    start_minute_idx/peak_minute_idx are NY minute-of-day at birth/peak, PRE-wrapped (a prior-day
    21:00-23:59 birth is negative: h*60+m-1440, so PRE's axis runs -180..570 instead of a
    21:00-23:59 tail stranded past its own 09:30 close - same convention TradingTool's
    sessionTimeChartRange already uses). peak_dev_abs/end_dev_abs are |dev_sig| at the peak tick
    and at the window-close mark, respectively - both None if the event never ticked (should not
    happen; a born event always ticks at least once, at birth). end_gap_pct is start_gap_pct's
    companion - Stack%-Bench% at the window-close mark instead of at birth - kept as its OWN
    field rather than derived: pnl_pct is built from start_stock/last_stock alone (see
    _pnl_pct), so end_gap_pct - start_gap_pct is NOT pnl_pct whenever the bench itself moved
    between birth and close (the normal case).
    Sparse: a (class, sign, bucket, window) cell is only present if it has at least
    one event; a cell's rate/avg_pnl fields are null (raw counts kept) below
    min_events_per_cell. `start_gap_pct` on a trade is the raw (unnormalized)
    Stack%-Bench% gap at entry - the "%" unit mode; `start_dev` is the sigma-
    normalized dev_sig - the "sigma" unit mode. Gamma/delta unit modes are ratios
    against that ticker's own static gamma/delta level from best_params.jsonl.gz.
    """
    import gc, gzip, json, math, time
    from collections import Counter, defaultdict
    from datetime import datetime, timedelta
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq

    CLASSES = ("pre", "open", "intra", "post")
    ALT_FROM = (int(pre_alt_from[0]), int(pre_alt_from[1])) if pre_alt_from else None
    # the streaming logic runs over TRACKS; rates and cells only ever see CLASSES
    TRACKS = CLASSES + (("pre_alt",) if ALT_FROM else ())
    PRE_TRACKS = ("pre",) + (("pre_alt",) if ALT_FROM else ())
    SIGNS = ("pos", "neg")
    WINDOWS = tuple(sorted(windows_days))
    MAX_WINDOW = WINDOWS[-1]

    def _open_text(path: str, mode: str = "wt"):
        if str(path).lower().endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    _math_isfinite = math.isfinite

    def is_finite_num(x) -> bool:
        # 7.2M calls per two row groups, and after the .tolist() conversions above almost every one
        # of them is a plain Python float. math.isfinite is exactly np.isfinite(float(x)) for that
        # type - same answer for NaN, +-inf and every finite value - and skips the float() call,
        # the numpy dispatch and the try/except frame. Anything else still takes the old path.
        if type(x) is float:
            return _math_isfinite(x)
        try:
            return np.isfinite(float(x))
        except Exception:
            return False

    def in_range(t, a, b):
        return (t is not None) and (a <= t <= b)

    def in_range_wrap(t, a, b):
        if t is None:
            return False
        if a <= b:
            return a <= t <= b
        return t >= a or t <= b

    # ---------------- windows (identical to v13 onefile) ----------------
    # PRE_START_TO is the BIRTH cutoff: 09:00 since 2026-11-08 (was 09:20). OPEN_FROM below is
    # still 09:20, so 09:00-09:20 takes no births in any class - see the v13 exporter's own
    # note on this; the two exporters must agree or their outputs cannot be compared.
    PRE_FROM, PRE_TO, PRE_START_TO = (21, 0), (9, 30), (9, 0)
    OPEN_FROM, OPEN_TO = (9, 20), (10, 0)
    INTRA_FROM, INTRA_TO = (9, 45), (16, 0)
    POST_FROM, POST_TO = (16, 0), (20, 0)

    CLASS_WINDOWS = {"pre": (PRE_FROM, PRE_TO), "open": (OPEN_FROM, OPEN_TO),
                      "intra": (INTRA_FROM, INTRA_TO), "post": (POST_FROM, POST_TO)}
    BIRTH_WINDOWS = {"pre": (PRE_FROM, PRE_START_TO), "open": (OPEN_FROM, OPEN_TO),
                      "intra": (INTRA_FROM, INTRA_TO), "post": (POST_FROM, POST_TO)}
    WRAP = {"pre": True, "open": False, "intra": False, "post": False}
    if ALT_FROM:
        CLASS_WINDOWS["pre_alt"] = (ALT_FROM, PRE_TO)          # same close as PRE, but nothing before ALT_FROM
        BIRTH_WINDOWS["pre_alt"] = (ALT_FROM, PRE_START_TO)
        WRAP["pre_alt"] = False
    PRE_AT_OPEN_TO = (9, 40)  # same grace bound v13 uses to finally close a PRE event

    IGNORE_WINDOWS = [((int(a[0]), int(a[1])), (int(b[0]), int(b[1]))) for a, b in ignore_windows]

    def minute_of_day(dt, cls: str) -> "int | None":
        if not isinstance(dt, datetime):
            return None
        total = dt.hour * 60 + dt.minute
        # PRE births only ever land in [21:00,23:59] (evening) or [00:00,09:40] (morning) -
        # noon is a safe splitter between the two.
        return total - 1440 if (cls == "pre" and total >= 720) else total

    def start_bin_label(abs_dev: float) -> str:
        if abs_dev >= start_bin_max:
            return f"{start_bin_max:.1f}+"
        v = max(float(dev_thr), float(abs_dev))
        idx = int((v - dev_thr) / start_bin_step + 1e-9)
        lo = dev_thr + idx * start_bin_step
        hi = lo + start_bin_step
        return f"{lo:.2f}-{hi:.2f}"

    # ---------------- pass 1: session-date ranking (file-relative windows) ----------------
    pf = pq.ParquetFile(input_path)
    date_col = "date" if "date" in pf.schema.names else None
    if date_col is None:
        raise KeyError("Input must contain a 'date' column")

    # The distinct dates, in Arrow. The shipped version materialised the whole column into pandas
    # and then ran astype(str).unique() over 2M values per row group just to learn ~800 dates;
    # pc.unique does it on the Arrow array and hands back only the distinct ones. Verified to
    # produce the identical set. (Row-group statistics cannot replace this: the file is sorted by
    # TICKER, so every group spans the entire date range and min/max tells us nothing about which
    # dates are actually present.)
    import pyarrow.compute as _pc
    seen_dates = set()
    for rg in range(pf.num_row_groups):
        _u = _pc.unique(pf.read_row_group(rg, columns=["date"]).column("date"))
        seen_dates.update(str(x) for x in _u.to_pylist() if x is not None)

    all_dates_sorted = sorted(seen_dates)  # ascending; ISO date strings sort correctly
    rank_from_end = {d: i for i, d in enumerate(reversed(all_dates_sorted))}  # 0 = most recent
    n_sessions = len(all_dates_sorted)
    recent_dates = all_dates_sorted[-MAX_WINDOW:]

    def windows_for_date(date_str: str):
        r = rank_from_end.get(str(date_str))
        if r is None:
            return ()
        return tuple(w for w in WINDOWS if r < w)

    print(f"sessions found: {n_sessions} (most recent: {all_dates_sorted[-1] if all_dates_sorted else None}), "
          f"windows={WINDOWS}")

    # ---------------- per-ticker aggregation state ----------------
    # agg[ticker][cls][sign][bucket][f"w{W}"] -> counts + dev_sum/n + pnl_sum/n
    def new_agg_ticker():
        return {c: {sk: defaultdict(lambda: defaultdict(
                        lambda: {"c": Counter(), "dev_sum": 0.0, "dev_n": 0, "pnl_sum": 0.0, "pnl_n": 0}))
                     for sk in SIGNS} for c in CLASSES}

    agg = {}
    static = {}   # ticker -> {"bench":..., "corr":..., "beta":..., "sigma":...}
    trades = {}   # ticker -> list of trade dicts (within the largest window only)

    def _spans(start_min, exit_min):
        # alive across a rollover: the entry baseline and the exit baseline may not be the same one
        if start_min is None or exit_min is None:
            return False
        return any(start_min < d <= exit_min for d in DISC_IDX)

    def record_event(ticker, cls, sign_key, start_date, start_dev, start_gap_pct, status, pnl_pct,
                      start_minute_idx=None, peak_minute_idx=None, peak_dev_abs=None, end_dev_abs=None,
                      end_gap_pct=None, exit_minute_idx=None, birth_gap_min=None, spans_disc=False):
        wins = windows_for_date(start_date)
        if not wins:
            return
        is_alt = cls == "pre_alt"          # the 04:02 track: trade log only, never a rate or cell statistic
        if not is_alt:
            abs_start_dev = abs(float(start_dev))
            bucket = start_bin_label(abs_start_dev)
            a = agg.setdefault(ticker, new_agg_ticker())
            for w in wins:
                cell = a[cls][sign_key][bucket][f"w{w}"]
                cell["c"][status] += 1
                cell["dev_sum"] += abs_start_dev
                cell["dev_n"] += 1
                if pnl_pct is not None:
                    cell["pnl_sum"] += pnl_pct
                    cell["pnl_n"] += 1

        if include_trade_log:
            pnl_usd = None if pnl_pct is None else round(position_usd * pnl_pct / 100.0, 2)
            trades.setdefault(ticker, []).append({
                "cls": "pre" if is_alt else cls, "sign": sign_key, "date": start_date,
                "start_dev": round(float(start_dev), 4),
                "start_gap_pct": None if start_gap_pct is None else round(float(start_gap_pct), 4),
                "status": status,
                "pnl_pct": None if pnl_pct is None else round(float(pnl_pct), 4),
                "pnl_usd": pnl_usd,
                "start_minute_idx": start_minute_idx,
                "peak_minute_idx": peak_minute_idx,
                "peak_dev_abs": None if peak_dev_abs is None else round(float(peak_dev_abs), 4),
                "end_dev_abs": None if end_dev_abs is None else round(float(end_dev_abs), 4),
                "end_gap_pct": None if end_gap_pct is None else round(float(end_gap_pct), 4),
                "exit_minute_idx": exit_minute_idx,
                "birth_gap_min": None if birth_gap_min is None else round(float(birth_gap_min), 1),
                "spans_disc": bool(spans_disc),
                **({"alt": True} if is_alt else {}),
            })

    # ---------------- event state (PRE/OPEN/INTRA/POST, symmetric) ----------------
    def _default_event_state():
        return {
            "active": False, "start_dt": None, "start_date": None, "start_dev": np.nan, "start_sign": 0,
            "start_stock": np.nan, "start_bench": np.nan,
            "last_stock": np.nan, "last_bench": np.nan, "last_dev_abs": np.nan,
            "last_dt": None, "birth_gap_min": None,
            "peak_abs": 0.0, "peak_dt": None,
            "hard_dt": None, "soft_found": False, "soft_dt": None,
        }

    events_by_ticker_cls = {}  # (ticker, cls) -> event state, only for the CURRENT ticker being streamed
    MIN_HOLD = max(1, int(min_hold_ticks))
    _disc_minutes = set()
    for _dh, _dm in discontinuities:
        for _k in range(int(disc_guard_min)):
            _disc_minutes.add(((_dh * 60 + _dm + _k) % 1440))
    def in_disc_guard(t):
        return (t[0] * 60 + t[1]) in _disc_minutes
    DISC_IDX = [(_dh * 60 + _dm) for _dh, _dm in discontinuities]   # on the PRE-wrapped axis (evening < 0)
    runs = {}                  # (ticker, cls) -> [consecutive qualifying ticks, sign]

    def _end_run(ticker, cls):
        # 13.9M calls per two row groups, and `runs` is empty for almost all of them: a signal has
        # to be mid-run for an entry to exist. Checking that first skips building the (ticker, cls)
        # tuple and the pop - popping from an empty dict does nothing anyway.
        if not runs:
            return
        runs.pop((ticker, cls), None)

    def get_event(ticker, cls):
        key = (ticker, cls)
        e = events_by_ticker_cls.get(key)
        if e is None:
            e = _default_event_state()
            events_by_ticker_cls[key] = e
        return e

    def start_event(ticker, cls, dt_now, ds, dev_now, stock_pct, bench_pct, birth_gap_min=None):
        e = _default_event_state()
        e["active"] = True
        e["birth_gap_min"] = birth_gap_min
        e["last_dt"] = dt_now
        e["start_dt"] = dt_now
        e["start_date"] = ds
        e["start_dev"] = float(dev_now)
        e["start_sign"] = 1 if float(dev_now) >= 0 else -1
        e["start_stock"] = stock_pct
        e["start_bench"] = bench_pct
        e["last_stock"] = stock_pct
        e["peak_abs"] = abs(float(dev_now))
        e["peak_dt"] = dt_now
        events_by_ticker_cls[(ticker, cls)] = e

    def process_tick(ticker, cls, dt_now, dev_now, stock_pct, bench_pct, t, _e=None):
        # The caller already holds the live event dict; looking it up a second time was ~3.5M
        # redundant lookups per two row groups. _e defaults to None so any other call site keeps
        # working unchanged.
        e = _e if _e is not None else get_event(ticker, cls)
        cur_abs = abs(float(dev_now))
        wf, wt = CLASS_WINDOWS[cls]
        inside = in_range_wrap(t, wf, wt) if WRAP[cls] else in_range(t, wf, wt)
        if not inside:
            return
        if is_finite_num(stock_pct):
            e["last_dt"] = dt_now  # WHEN that mark was actually printed: thin names stop printing
            # hours before the window closes, so "exit at 09:30" is really "exit at the last print"
            e["last_stock"] = stock_pct  # mark-to-window-close: the exit basis for EVERY
            # outcome (hard/soft/none alike) - we hold to the class's own close (09:30 PRE,
            # 10:00 OPEN, 16:00 INTRA, 20:00 POST), not to whichever tick first normalized.
        if is_finite_num(bench_pct):
            e["last_bench"] = bench_pct  # same mark-to-window-close, pairs with last_stock
            # for end_gap_pct - the "% at the end of the event" companion to start_gap_pct.
        e["last_dev_abs"] = cur_abs  # same mark-to-window-close, for the |dev_sig| axis
        if cur_abs > float(e["peak_abs"]):
            e["peak_abs"] = float(cur_abs)
            e["peak_dt"] = dt_now
            if e["hard_dt"] is None:
                e["soft_found"] = False
                e["soft_dt"] = None
        if e["hard_dt"] is None and cur_abs <= norm_thr:
            e["hard_dt"] = dt_now
        if e["hard_dt"] is None and isinstance(e["peak_dt"], datetime) and e["peak_abs"] > 0:
            if dt_now >= e["peak_dt"] and cur_abs <= (float(e["peak_abs"]) / float(soft_ratio)):
                if (not e["soft_found"]) or (dt_now < e["soft_dt"]):
                    e["soft_found"] = True
                    e["soft_dt"] = dt_now

    def _pnl_pct(sign_key, start_stock, exit_stock):
        if not (is_finite_num(start_stock) and is_finite_num(exit_stock)):
            return None
        diff = float(exit_stock) - float(start_stock)
        return -diff if sign_key == "pos" else diff  # SHORT on pos, LONG on neg

    def finalize_event(ticker, cls):
        e = get_event(ticker, cls)
        if not e["active"]:
            return
        if e["hard_dt"] is not None:
            status = "hard"
        elif e["soft_found"] and e["soft_dt"] is not None:
            status = "soft"
        else:
            status = "none"
        sk = "pos" if e["start_sign"] > 0 else "neg"

        # exit is always the window-close mark (09:30 PRE / 10:00 OPEN / 16:00 INTRA /
        # 20:00 POST), for hard/soft/none alike - see the comment in process_tick.
        pnl_pct = _pnl_pct(sk, e["start_stock"], e["last_stock"])
        start_gap_pct = None
        if is_finite_num(e["start_stock"]) and is_finite_num(e["start_bench"]):
            start_gap_pct = float(e["start_stock"]) - float(e["start_bench"])
        end_gap_pct = None
        if is_finite_num(e["last_stock"]) and is_finite_num(e["last_bench"]):
            end_gap_pct = float(e["last_stock"]) - float(e["last_bench"])

        record_event(
            ticker, cls, sk, e["start_date"], e["start_dev"], start_gap_pct, status, pnl_pct,
            start_minute_idx=minute_of_day(e["start_dt"], cls),
            peak_minute_idx=minute_of_day(e["peak_dt"], cls),
            peak_dev_abs=e["peak_abs"],
            end_dev_abs=e["last_dev_abs"] if is_finite_num(e["last_dev_abs"]) else None,
            end_gap_pct=end_gap_pct,
            exit_minute_idx=minute_of_day(e["last_dt"], cls),
            birth_gap_min=e["birth_gap_min"],
            spans_disc=_spans(minute_of_day(e["start_dt"], cls), minute_of_day(e["last_dt"], cls)),
        )
        events_by_ticker_cls[(ticker, cls)] = _default_event_state()

    def on_new_day(ticker):
        for _c in TRACKS:
            _end_run(ticker, _c)
        for _pc in PRE_TRACKS:
            e_pre = get_event(ticker, _pc)
            if e_pre["active"]:
                start_t = (e_pre["start_dt"].hour, e_pre["start_dt"].minute) if isinstance(e_pre["start_dt"], datetime) else None
                started_overnight = start_t is not None and start_t >= PRE_FROM
                if not started_overnight:
                    finalize_event(ticker, _pc)
        for cls in ("open", "intra", "post"):
            if get_event(ticker, cls)["active"]:
                finalize_event(ticker, cls)

    # ---------------- main streaming pass ----------------
    cur_ticker = None
    cur_day = None

    def flush_ticker(ticker):
        for _c in TRACKS:
            _end_run(ticker, _c)
        for cls in TRACKS:
            if get_event(ticker, cls)["active"]:
                finalize_event(ticker, cls)
        for cls in TRACKS:
            events_by_ticker_cls.pop((ticker, cls), None)

    prev_state = {"tk": None, "dt": None}   # previous valid tick of the SAME ticker

    def process_chunk(chunk: "pd.DataFrame"):
        nonlocal cur_ticker, cur_day

        req = {"ticker", "date", "dt", "dev_sig"}
        if not req.issubset(chunk.columns):
            raise KeyError(f"Input must contain columns: {sorted(req)}")

        def col(name):
            return chunk[name] if name in chunk.columns else pd.Series(np.nan, index=chunk.index)

        s_dt_ts = pd.to_datetime(col("dt"), errors="coerce", utc=True)
        s_dev = pd.to_numeric(col("dev_sig"), errors="coerce")

        dt_ok = s_dt_ts.notna()
        dev_ok = np.isfinite(s_dev.to_numpy(dtype="float64", copy=False))
        mask = (dt_ok.to_numpy(copy=False) & dev_ok)

        if mask.any():
            dt2 = s_dt_ts[mask]
            h = dt2.dt.hour.to_numpy(dtype="int16", copy=False)
            m = dt2.dt.minute.to_numpy(dtype="int16", copy=False)

            def _in_win(h, m, a, b):
                ah, am = a; bh, bm = b
                return ((h > ah) | ((h == ah) & (m >= am))) & ((h < bh) | ((h == bh) & (m <= bm)))

            ig = np.zeros_like(h, dtype=bool)
            for a, b in IGNORE_WINDOWS:
                ig |= _in_win(h, m, a, b)
            idx_mask = np.flatnonzero(mask)
            mask[idx_mask] = ~ig

        if not mask.any():
            return

        # .tolist() on each of these is what the row loop below actually wants: indexing a numpy
        # array boxes a fresh numpy scalar per read, and np.float64 arithmetic is several times
        # slower than plain float. The VALUES are unchanged - tolist() is an exact conversion for
        # float64/int16/object arrays, NaN included.
        dt_py = list(s_dt_ts[mask].dt.to_pydatetime())
        tk_arr = col("ticker")[mask].to_numpy(copy=False).tolist()
        ds_arr = col("date")[mask].to_numpy(copy=False).tolist()
        dev_arr = s_dev[mask].to_numpy(dtype="float64", copy=False).tolist()

        s_bench_name = col("bench")
        s_corr = col("corr"); s_beta = col("beta"); s_sigma = col("sigma")
        bench_name_arr = s_bench_name[mask].to_numpy(copy=False) if "bench" in chunk.columns else None
        corr_arr = s_corr[mask].to_numpy(copy=False) if "corr" in chunk.columns else None
        beta_arr = s_beta[mask].to_numpy(copy=False) if "beta" in chunk.columns else None
        sigma_arr = s_sigma[mask].to_numpy(copy=False) if "sigma" in chunk.columns else None

        s_bench_num = (pd.to_numeric(col(BENCH_NUM_FIELD), errors="coerce")
                       if BENCH_NUM_FIELD in chunk.columns else pd.Series(np.nan, index=chunk.index))
        s_stock_pct = (pd.to_numeric(col(STOCK_NUM_FIELD), errors="coerce")
                       if STOCK_NUM_FIELD in chunk.columns else pd.Series(np.nan, index=chunk.index))
        bench_num_arr = s_bench_num[mask].to_numpy(dtype="float64", copy=False).tolist()
        stock_arr = s_stock_pct[mask].to_numpy(dtype="float64", copy=False).tolist()

        dt2 = s_dt_ts[mask]
        h_arr = dt2.dt.hour.to_numpy(dtype="int16", copy=False).tolist()
        m_arr = dt2.dt.minute.to_numpy(dtype="int16", copy=False).tolist()

        n = len(dev_arr)
        for i in range(n):
            tk = tk_arr[i]
            ds = ds_arr[i]
            dt_now = dt_py[i]
            dev_now = dev_arr[i]
            stock_pct = stock_arr[i]
            bench_num = bench_num_arr[i]
            t = (h_arr[i], m_arr[i])   # already python ints - see the .tolist() note above

            gap_prev = None
            if prev_state["tk"] == tk and prev_state["dt"] is not None:
                gap_prev = (dt_now - prev_state["dt"]).total_seconds() / 60.0
            prev_state["tk"], prev_state["dt"] = tk, dt_now

            if cur_ticker is not None and tk != cur_ticker:
                flush_ticker(cur_ticker)
                cur_ticker, cur_day = tk, ds
            if cur_ticker is None:
                cur_ticker, cur_day = tk, ds

            # setdefault builds its default on EVERY call, so the shipped line allocated a fresh
            # four-key dict per row - 1.8M throwaway dicts per row group - and used it only on the
            # first row of each ticker.
            st = static.get(tk)
            if st is None:
                st = static[tk] = {"bench": None, "corr": None, "beta": None, "sigma": None}
            if st["bench"] is None and bench_name_arr is not None:
                bn = bench_name_arr[i]
                if pd.notna(bn) and str(bn).strip():
                    st["bench"] = str(bn)
            if st["corr"] is None and corr_arr is not None and beta_arr is not None and sigma_arr is not None:
                c, b, s = corr_arr[i], beta_arr[i], sigma_arr[i]
                if pd.notna(c) and pd.notna(b) and pd.notna(s):
                    st["corr"], st["beta"], st["sigma"] = c, b, s

            if ds != cur_day:
                cur_day = ds
                on_new_day(tk)

            for _pc in PRE_TRACKS:
                e_pre = get_event(tk, _pc)
                if e_pre["active"] and isinstance(e_pre["start_dt"], datetime):
                    _pre_start_t = (e_pre["start_dt"].hour, e_pre["start_dt"].minute)
                    _close_dt = e_pre["start_dt"].replace(hour=PRE_AT_OPEN_TO[0], minute=PRE_AT_OPEN_TO[1], second=0, microsecond=0)
                    if _pre_start_t > PRE_AT_OPEN_TO:
                        _close_dt = _close_dt + timedelta(days=1)
                    if dt_now > _close_dt:
                        finalize_event(tk, _pc)

            for cls in ("open", "intra", "post"):
                if get_event(tk, cls)["active"] and (t > CLASS_WINDOWS[cls][1]):
                    finalize_event(tk, cls)

            for cls in TRACKS:
                e = get_event(tk, cls)
                if not e["active"]:
                    bf, bt = BIRTH_WINDOWS[cls]
                    born_inside = in_range_wrap(t, bf, bt) if WRAP[cls] else in_range(t, bf, bt)
                    if born_inside and (not in_disc_guard(t)) and abs(float(dev_now)) >= dev_thr:
                        sg = 1 if float(dev_now) >= 0 else -1
                        r = runs.get((tk, cls))
                        if r is not None and r[1] == sg:
                            r[0] += 1
                        else:
                            _end_run(tk, cls)      # the other sign cut the previous run short
                            r = runs[(tk, cls)] = [1, sg, gap_prev]   # gap_prev = silence before the FIRST tick
                        if r[0] >= MIN_HOLD:
                            start_event(tk, cls, dt_now, ds, dev_now, stock_pct, bench_num, birth_gap_min=r[2])
                            # start_event REBINDS the dict, so the `e` fetched above is now stale -
                            # this is the one branch that has to look it up again.
                            e = get_event(tk, cls)
                            runs.pop((tk, cls), None)
                    else:
                        _end_run(tk, cls)
                if e["active"]:
                    process_tick(tk, cls, dt_now, dev_now, stock_pct, bench_num, t, _e=e)

    t0 = time.time()
    total_rows = 0
    wanted = ["ticker", "date", "dt", "dev_sig", "bench", "corr", "beta", "sigma", STOCK_NUM_FIELD, BENCH_NUM_FIELD]
    cols = [c for c in wanted if c in pf.schema.names]
    for rg in range(pf.num_row_groups):
        chunk = pf.read_row_group(rg, columns=cols).to_pandas()
        process_chunk(chunk)
        total_rows += len(chunk)
        if (rg + 1) % parquet_batch_log_every == 0:
            print(f"[rg {rg+1}/{pf.num_row_groups}] rows={total_rows:,} elapsed={time.time()-t0:,.1f}s")
        del chunk
        if (rg + 1) % max(10, parquet_batch_log_every * 2) == 0:
            gc.collect()

    if cur_ticker is not None:
        flush_ticker(cur_ticker)

    # ---------------- build sparse output ----------------
    def cell_summary(cell):
        c = cell["c"]
        total = int(sum(c.values()))
        hard = int(c.get("hard", 0)); soft = int(c.get("soft", 0)); none = int(c.get("none", 0))
        any_ = hard + soft
        enough = total >= min_events_per_cell
        return {
            "total": total, "hard": hard, "soft": soft, "none": none,
            "rate_any": (any_ / total) if (total and enough) else None,
            "rate_hard": (hard / total) if (total and enough) else None,
            "rate_soft": (soft / total) if (total and enough) else None,
            "avg_start_dev": round(cell["dev_sum"] / cell["dev_n"], 4) if cell["dev_n"] else None,
            "avg_pnl_pct": round(cell["pnl_sum"] / cell["pnl_n"], 4) if (cell["pnl_n"] and enough) else None,
            "avg_pnl_usd": round(position_usd * (cell["pnl_sum"] / cell["pnl_n"]) / 100.0, 2) if (cell["pnl_n"] and enough) else None,
        }

    rows = []
    for ticker, cls_map in agg.items():
        classes_out = {}
        ticker_total = 0
        for cls in CLASSES:
            sign_out = {}
            for sk in SIGNS:
                buckets = cls_map[cls][sk]
                if not buckets:
                    continue
                bucket_out = {}
                for bucket, win_map in buckets.items():
                    win_out = {}
                    for wkey, cell in win_map.items():
                        summ = cell_summary(cell)
                        win_out[wkey] = summ
                        if wkey == f"w{MAX_WINDOW}":
                            ticker_total += summ["total"]
                    bucket_out[bucket] = win_out
                if bucket_out:
                    sign_out[sk] = bucket_out
            if sign_out:
                classes_out[cls] = sign_out

        if ticker_total < int(min_events_per_ticker):
            continue

        st = static.get(ticker, {})
        row = {
            "ticker": ticker,
            "bench": st.get("bench"),
            "corr": None if st.get("corr") is None else float(st["corr"]),
            "beta": None if st.get("beta") is None else float(st["beta"]),
            "sigma": None if st.get("sigma") is None else float(st["sigma"]),
            "classes": classes_out,
        }
        if include_trade_log:
            row["trades"] = sorted(trades.get(ticker, []), key=lambda r: (r["date"], r["cls"]))
        rows.append(row)

    payload = {
        "meta": {
            "version": "v2-rolling-perf-pnl",
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "windows_days": list(WINDOWS),
            "sessions_seen": n_sessions,
            "most_recent_session": all_dates_sorted[-1] if all_dates_sorted else None,
            "recent_dates": recent_dates,  # oldest->newest, length == MAX_WINDOW; use to re-slice trades into w5/w20/w40 client-side
            "dev_thr": dev_thr, "norm_thr": norm_thr, "soft_ratio": soft_ratio,
            "start_bin_step": start_bin_step, "start_bin_max": start_bin_max,
            "bucketed_on": "start_deviation (birth |dev_sig|, no lookahead)",
            "min_events_per_ticker": min_events_per_ticker,
            "min_events_per_cell": min_events_per_cell,
            "position_usd": position_usd,
            "min_hold_ticks": MIN_HOLD,
            "pre_alt_from": list(ALT_FROM) if ALT_FROM else None,
            "pnl_basis": "real Stack% move, entry->class window-close (09:30 PRE/10:00 OPEN/16:00 INTRA/20:00 POST) regardless of hard/soft/none; SHORT on pos, LONG on neg",
        },
        "rows": rows,
    }

    with _open_text(output_path, "wt") as f:
        json.dump(payload, f, ensure_ascii=False)

    print(f"DONE rows_read={total_rows:,} tickers_out={len(rows):,} -> {output_path}")


In [ ]:
# ------------------------------------------------------------------
# ROLLING RECENCY PERFORMANCE (last 5/20/40/65 sessions) + P&L TRADE LOG, split by
# class x sign x start-deviation bucket. Feeds the "Scout" UI: pick ticker/class/
# sign/window, set a "start" threshold in %, sigma, gamma- or delta-relative units
# (gamma/delta come from best_params.jsonl.gz, joined by ticker) and get both a
# hit-rate and a 000-position P&L curve, entirely client-side against this file.
# ------------------------------------------------------------------

ROLLING_OUT_PATH = OUT_DIR / "rolling_perf.json.gz"

devsig_rolling_perf_exporter(
    input_path=str(FINAL_PATH),
    output_path=str(ROLLING_OUT_PATH),

    windows_days=(5, 20, 40, 65),

    dev_thr=0.60,     # start: 0.6 sigma (dev_sig is already in sigma units)
    norm_thr=0.20,
    soft_ratio=3.0,
    ignore_windows=(((3, 58), (4, 2)), ((7, 58), (8, 2))),   # dropped ALWAYS: no situation starts there
    pre_alt_from=(4, 2),   # the "FROM 04:02" model: a second PRE track whose births start there (trade log only)
    min_hold_ticks=2,   # a signal must hold two consecutive ticks; 1 = the old single-tick birth

    start_bin_step=0.2,
    start_bin_max=3.0,

    min_events_per_ticker=10,
    min_events_per_cell=2,

    position_usd=1000.0,
    include_trade_log=True,
)

print("Rolling performance + P&L export completed ->", ROLLING_OUT_PATH)
